# R2 · Reranker + entities (Rathi)

**Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**. It takes about 1–1.5 hours; keep the tab open.

Before running, add these in **Colab Secrets** (the key icon on the left) and switch on notebook access:
- `GITHUB_TOKEN`: a GitHub token with `repo` access, from the account that pushes (rathiabeth-sudo)
- `HF_TOKEN`: a Hugging Face token. It needs **write** access for step 6.
- `WANDB_KEY`: a Weights & Biases API key

Steps: 1 setup → 2 10-row checks → 3 fine-tune → 4 score all snippets → 5 entities → 6 save the model → 7 push the outputs to the `rathi/R2-reranker` branch.

In [18]:
# 1. Setup. The repo is public, so cloning needs no token; GITHUB_TOKEN is only used to push at the end.
from google.colab import userdata
import os
REPO = '/content/twincheck'
if not os.path.exists(REPO):
    !git clone https://github.com/himaallu/Group-Project.git {REPO}
%cd {REPO}
!git fetch -q origin && git checkout -q rathi/R2-reranker && git pull -q origin rathi/R2-reranker
!git config user.name "Rathi"
!git config user.email "335395537+rathiabeth-sudo@users.noreply.github.com"

# Check the GitHub token now, not after an hour of training.
GH = userdata.get('GITHUB_TOKEN').strip()
!git remote set-url --push origin https://{GH}@github.com/himaallu/Group-Project.git
ok = !git ls-remote https://{GH}@github.com/himaallu/Group-Project.git HEAD
print('GitHub token OK' if ok and 'fatal' not in ' '.join(ok) else 'GITHUB_TOKEN rejected: make a new one with the "repo" box ticked')

# Colab already has torch, transformers, pandas, pyarrow and spaCy. Only add or upgrade what R2 needs.
!pip install -q -U wandb

# spaCy model: en_core_web_trf (CLAUDE.md); fall back to en_core_web_lg if trf cannot install on this Python.
import subprocess, sys
SPACY = 'en_core_web_trf'
if subprocess.run([sys.executable, '-m', 'spacy', 'download', SPACY]).returncode != 0 or \
        subprocess.run([sys.executable, '-c', f'import spacy; spacy.load("{SPACY}")']).returncode != 0:
    SPACY = 'en_core_web_lg'
    subprocess.run([sys.executable, '-m', 'spacy', 'download', SPACY], check=True)
print('spaCy model:', SPACY)

import wandb
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN').strip()
wandb.login(key=userdata.get('WANDB_KEY').strip(), relogin=True)   # error here = copy the key again from wandb.ai/authorize
!nvidia-smi --query-gpu=name,memory.total --format=csv

/content/twincheck
GitHub token OK
spaCy model: en_core_web_trf


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.forge.coreweave.com.
wandb: Appending key for api.forge.coreweave.com to your netrc file: /root/.netrc


name, memory.total [MiB]
NVIDIA A100-SXM4-80GB, 81920 MiB


## 2. 10-row checks (CLAUDE.md rule 3). Every command here must finish without errors before the full run.

In [19]:
!python -m pytest tests/retrieval tests/data -q
!python -m src.retrieval.train_reranker --n-train-claims 5 --n-val-claims 5 --no-wandb     --out models/test10/reranker --metrics models/test10/eval.json --ckpt-dir /content/ckpt_test10
!python -m src.retrieval.rerank data/processed/en/multifc_val.jsonl --limit 10     --model models/test10/reranker --out features/test10/relevance_en.parquet
!python -m src.data.entities data/processed/en/multifc_val.jsonl --limit 10     --out features/test10/entities_en.jsonl --model {SPACY}
import pandas as pd, json
print(pd.read_parquet('features/test10/relevance_en.parquet').head(10))
print(open('features/test10/entities_en.jsonl', encoding='utf-8').readline()[:600])

................                                                         [100%]
16 passed in 0.03s
10 train pairs, 10 val pairs, device cuda
Loading weights: 100% 393/393 [00:00<00:00, 6467.46it/s]
zero-shot: {'n_pairs': 10, 'accuracy': 0.7, 'auroc': 1.0, 'mean_relevance_pos': 0.46449637319892645, 'mean_relevance_neg': 0.00020014567126054318}
/content/twincheck/src/retrieval/train_reranker.py:174: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()
step 1/1 loss 1.5723 (0s)
fine-tuned: {'n_pairs': 10, 'accuracy': 0.7, 'auroc': 1.0, 'mean_relevance_pos': 0.46449637319892645, 'mean_relevance_neg': 0.00020014567126054318}
Writing model sh

## 3. Fine-tune the reranker: 20k pairs, 1 epoch, logged to W&B (~10–15 min on a T4)

In [24]:
!git pull -q origin rathi/R2-reranker

In [25]:
import torch
import os

if not torch.cuda.is_available():
    print("❌ WARNING: GPU is not available! Please change your runtime type to T4 GPU (Runtime -> Change runtime type) before running this cell, otherwise training on CPU will take several hours.")
else:
    print("✅ GPU detected:", torch.cuda.get_device_name(0))
    !python -m src.retrieval.train_reranker
    if os.path.exists('results/R2/reranker_eval.json'):
        print(open('results/R2/reranker_eval.json').read())
    else:
        print("❌ Error: Evaluation file was not created. Check training logs above for issues.")

✅ GPU detected: NVIDIA A100-SXM4-80GB
20000 train pairs, 2000 val pairs, device cuda
Loading weights: 100% 393/393 [00:00<00:00, 6083.75it/s]
wandb: [wandb.login()] Loaded credentials for https://api.forge.coreweave.com from /root/.netrc.
wandb: Currently logged in as: rathiabeth (rathiabeth-uowd) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin
wandb: Using an existing wandb-core service via WANDB_SERVICE.
wandb: ⢿ setting up run lhb3pemh (0.0s)
wandb: ⣻ setting up run lhb3pemh (0.0s)
wandb: ⣽ setting up run lhb3pemh (0.0s)
wandb: ⣾ setting up run lhb3pemh (0.0s)
wandb: ⣷ setting up run lhb3pemh (0.5s)
wandb: ⣯ setting up run lhb3pemh (0.5s)
wandb: ⣟ setting up run lhb3pemh (0.5s)
wandb: ⡿ setting up run lhb3pemh (0.5s)
wandb: ⢿ setting up run lhb3pemh (0.5s)
wandb: ⣻ setting up run lhb3pemh (1.0s)
wandb: Tracking run with wandb version 0.31.0
wandb: Run data is saved locally in /content/twincheck/wandb/run-20261117_123222-lhb3pemh
wandb: Run `wandb off

## 4. Score every (claim, snippet) pair in all English splits → `features/relevance_en.parquet` (~20 min)

In [21]:
EN = 'data/processed/en'
FILES = ' '.join(f'{EN}/{d}_{s}.jsonl' for d in ('multifc', 'averitec') for s in ('train', 'val', 'test'))
!python -m src.retrieval.rerank {FILES} --model models/reranker_v1 --out features/relevance_en.parquet
rel = pd.read_parquet('features/relevance_en.parquet')
print(rel.describe())
print('share of snippets with relevance > 0.3:', round((rel.relevance > 0.3).mean(), 3))

Loading weights: 100% 393/393 [00:00<00:00, 6190.31it/s]
data/processed/en/multifc_train.jsonl: 16847 records, 123233 snippets (81s on cuda)
data/processed/en/multifc_val.jsonl: 2104 records, 15364 snippets (10s on cuda)
data/processed/en/multifc_test.jsonl: 2106 records, 15366 snippets (10s on cuda)
data/processed/en/averitec_train.jsonl: 2309 records, 5675 snippets (4s on cuda)
data/processed/en/averitec_val.jsonl: 252 records, 625 snippets (0s on cuda)
data/processed/en/averitec_test.jsonl: 429 records, 1112 snippets (1s on cuda)
161375 rows -> features/relevance_en.parquet; mean relevance 0.972
         snippet_idx      relevance
count  161375.000000  161375.000000
mean        3.321320       0.972397
std         2.438442       0.149498
min         0.000000       0.000873
25%         1.000000       0.999558
50%         3.000000       0.999886
75%         5.000000       0.999949
max         9.000000       0.999962
share of snippets with relevance > 0.3: 0.976


## 5. spaCy entities → `features/entities_en.jsonl` (~30 min)

In [22]:
!python -m src.data.entities {FILES} --out features/entities_en.jsonl --model {SPACY}
!ls -la features/

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/thinc/util.py:401: VisibleDeprecationWarning: This function is deprecated and will be removed in a future release. Use the cupy.from_dlpack() array constructor instead.
  dlpack_tensor = xp_tensor.toDlpack()  # type: ignore
data/processed/en/multifc_train.jsonl: 16847 records (374s)
data/processed/en/multifc_val.jsonl: 2104 records (47s)
data/processed/en/multifc_test.jsonl: 2106 records (46s)
data/processed/en/averitec_train.jsonl: 2309 records (26s)
data/processed/en/averitec_val.jsonl: 252 records (3s)
data/processed/en/averitec_test.jsonl: 429 records (5s)
24047 records -> features/entities_en.jsonl
total 49072
drwxr-xr-x  3 root root     4096 Nov 17 11:54 .
drwxr-xr-x 15 root root     4096 Nov 17 11:49 ..
-rw-r--r--  1 root root 49599376 Nov 17 12:31 entities_

## 6. Save the fine-tuned model

The model is about 1.1 GB, which is more than GitHub's free Git LFS quota (1 GB for the whole repo). So it goes to **Hugging Face Hub** instead, and git keeps a small `models/reranker_v1/README.md` that points to it. Teammates load it with `Reranker("<hub id>")`.

In [16]:
from huggingface_hub import HfApi
api = HfApi(token=os.environ['HF_TOKEN'])
HUB_ID = f"{api.whoami()['name']}/twincheck-reranker-v1"
api.create_repo(HUB_ID, exist_ok=True, private=False)
api.upload_folder(folder_path='models/reranker_v1', repo_id=HUB_ID,
                  commit_message='R2: bge-reranker-v2-m3 fine-tuned on MultiFC + AVeriTeC relevance pairs')
ev = json.load(open('results/R2/reranker_eval.json'))
open('models/reranker_v1/README.md', 'w').write(
    f"# reranker_v1 (R2)

Weights are on Hugging Face Hub: https://huggingface.co/{HUB_ID} "
    f"(too big for git).

Load: `from src.retrieval.rerank import Reranker; rr = Reranker('{HUB_ID}')`

"
    f"Base `{ev['base_model']}`, 1 epoch, {ev['n_train_pairs']} pairs, lr {ev['lr']}, seed {ev['seed']}. "
    f"Val AUROC {ev['val_zero_shot']['auroc']:.3f} -> {ev['val_fine_tuned']['auroc']:.3f}. W&B: {ev['wandb_url']}
")
print(open('models/reranker_v1/README.md').read())

# reranker_v1 (R2)

Weights are on Hugging Face Hub: https://huggingface.co/RathiAbeth/twincheck-reranker-v1 (too big for git).

Load: `from src.retrieval.rerank import Reranker; rr = Reranker('RathiAbeth/twincheck-reranker-v1')`

Base `BAAI/bge-reranker-v2-m3`, 1 epoch, 20000 pairs, lr 2e-05, seed 42.
Val AUROC 0.980 -> 0.998. W&B: https://forge.coreweave.com/wandb/rathiabeth-uowd/twincheck/runs/yftu2n5c



## 7. Push the outputs to the `rathi/R2-reranker` branch

In [23]:
!git config user.name "Rathi"
!git config user.email "335395537+rathiabeth-sudo@users.noreply.github.com"
!du -h features/relevance_en.parquet features/entities_en.jsonl
# GitHub refuses files over 100 MB; split entities by dataset if this fails.
!git add features/relevance_en.parquet features/entities_en.jsonl models/reranker_v1/README.md results/R2/reranker_eval.json
!git commit -q -m "[R2] Add relevance scores, entities and reranker eval from Colab run"
!git push -q origin rathi/R2-reranker && echo PUSHED

620K	features/relevance_en.parquet
48M	features/entities_en.jsonl
PUSHED


**Done.** Copy the output of steps 3–5 (or just the W&B link) back to Claude. Then **Runtime → Disconnect and delete runtime**.